# Evaluation Harness — BRFSS GenHlth

Builds the reusable pieces every algorithm notebook (DT, kNN, SVM, NN-sklearn, NN-PyTorch) will load: the single train/test split, leakage-safe preprocessing, sanity baselines, and one shared metrics function.

Input: `data/sample_20k.csv` (produced by `eda.ipynb`).

## Step 1: Load the sample and take one held-out train/test split

This split is touched exactly once per model, at final evaluation. Everything before that (tuning, model-complexity curves, learning curves) happens only on the training portion, via cross-validation or a further validation split — never on `X_test`/`y_test`.

In [1]:
import numpy as np, pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

SEED = 42
TARGET = "GenHlth"

sample = pd.read_csv("data/sample_20k.csv")
FEATURES = [c for c in sample.columns if c != TARGET]

X = sample[FEATURES]
y = sample[TARGET].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEED
)

assert set(X_train.index).isdisjoint(X_test.index)
assert TARGET not in X_train.columns
X_train.shape, X_test.shape

((16000, 21), (4000, 21))

In [2]:
Path("results").mkdir(exist_ok=True)
np.save("results/train_idx.npy", X_train.index.values)
np.save("results/test_idx.npy", X_test.index.values)

def class_table(y):
    c = y.value_counts().sort_index()
    return pd.DataFrame({"count": c, "share_%": (c / len(y) * 100).round(2)})

pd.concat(
    [class_table(y_train).add_prefix("train_"), class_table(y_test).add_prefix("test_")],
    axis=1,
)

,train_count,train_share_%,test_count,test_share_%
GenHlth,,,,
1,2857,17.86,714,17.85
2,5619,35.12,1404,35.10
3,4771,29.82,1193,29.83
4,1991,12.44,498,12.45
5,762,4.76,191,4.78


## Step 2: Leakage-safe preprocessing

The FAQ's leakage rule: never fit a scaler (or any preprocessing) on data outside the current training fold. Fitting a `StandardScaler` on the full sample before splitting would let the test set's mean/std quietly influence how training data gets scaled — a small but real form of leakage.

The fix is mechanical, not conceptual: **fit on `X_train`, transform both `X_train` and `X_test` using that same fitted scaler.** We demonstrate this manually once below (and prove it numerically — the test set's scaled mean will *not* land exactly on 0, because it was standardized using training statistics, not its own), then define the reusable pattern every algorithm notebook will use: wrap the scaler and the estimator together in a single `sklearn.Pipeline`. Wrapped this way, calling `.fit()` on any training fold — whether it's the full training set or one fold inside cross-validation — automatically refits the scaler on just that fold, so the same leakage rule holds during tuning too, not just at the final split.

One feature-treatment decision to state explicitly: `Diabetes_012` (0 = no diabetes, 1 = prediabetes, 2 = diabetes) is treated as ordinal, consistent with `Age`/`Education`/`Income`, since its levels represent increasing severity rather than unrelated categories. All 21 features are scaled uniformly. This has no effect on the Decision Tree (its splits are invariant to monotonic scaling) and is required for kNN, SVM, and the neural networks.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("train scaled mean (should be ~0.0):   ", X_train_scaled.mean(axis=0)[:3].round(4))
print("test scaled mean (should NOT be 0.0): ", X_test_scaled.mean(axis=0)[:3].round(4))

In [ ]:
from sklearn.pipeline import make_pipeline

def scaled_pipeline(estimator):
    """Wrap a scale-sensitive estimator (kNN, SVM, NN) so preprocessing
    is always refit on whatever training fold .fit() is called with --
    never on validation/test data, even inside cross-validation."""
    return make_pipeline(StandardScaler(), estimator)

## Step 3: Sanity baselines

Before trusting any real model's score, we need to know what "no skill" looks like on this exact split and metric set — otherwise a mediocre model can look impressive just by beating nothing in particular. Two checks:

1. **Dummy (most-frequent) classifier** — always predicts the majority class (2). This is the absolute floor: every real model must clearly beat it, especially on Macro-F1 and balanced accuracy, which a most-frequent strategy is specifically bad at (it scores 0 recall on every class except the one it always predicts).
2. **Shuffled-label run** — fit a real, trainable model (Logistic Regression, via the same `scaled_pipeline` helper from step 2) on `y_train` with the row order randomly permuted, so any true relationship between `X` and `y` is destroyed, then evaluate it against the *real* `y_test`. Performance should collapse to roughly the dummy's level. If a shuffled-label model still scores well, that's a direct sign of leakage somewhere in the pipeline (e.g., the split isn't actually disjoint, or a feature is secretly encoding the target).

We also fit the same Logistic Regression on the real (unshuffled) labels as a reference point — not because it's one of the four required algorithms, but to confirm the harness can actually detect a real signal when one exists, not just fail safely when one doesn't.

In [5]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, balanced_accuracy_score

def evaluate(name, estimator, y_fit):
    """Fit on (X_train, y_fit), score against the real held-out test set."""
    estimator.fit(X_train, y_fit)
    pred = estimator.predict(X_test)
    return {
        "model": name,
        "accuracy": accuracy_score(y_test, pred),
        "macro_F1": f1_score(y_test, pred, average="macro"),
        "balanced_acc": balanced_accuracy_score(y_test, pred),
    }

rng = np.random.default_rng(SEED)
y_train_shuffled = rng.permutation(y_train.values)

results = [
    evaluate("Dummy (most frequent)", DummyClassifier(strategy="most_frequent"), y_train),
    evaluate("Logistic Regression, shuffled labels", scaled_pipeline(LogisticRegression(max_iter=500)), y_train_shuffled),
    evaluate("Logistic Regression, real labels (reference)", scaled_pipeline(LogisticRegression(max_iter=500)), y_train),
]
pd.DataFrame(results).round(3)

,model,accuracy,macro_F1,balanced_acc
0,Dummy (most frequent),0.351,0.104,0.200
1,"Logistic Regression, shuffled labels",0.361,0.128,0.208
2,"Logistic Regression, real labels (reference)",0.468,0.417,0.400


## Step 4: Tuning protocol

We validate the protocol below by cross-validating the reference Logistic Regression from step 3 and checking that its CV Macro-F1 lines up with the 0.417 it scored on the real held-out test set.

In [6]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
CV_SCORING = "f1_macro"

cv_scores = cross_val_score(
    scaled_pipeline(LogisticRegression(max_iter=500)),
    X_train, y_train, cv=CV, scoring=CV_SCORING,
)
print("per-fold Macro-F1:", cv_scores.round(3))
print("mean Macro-F1:    ", cv_scores.mean().round(3), "+/-", cv_scores.std().round(3))
print("held-out test Macro-F1 (from step 3): 0.417")

per-fold Macro-F1: [0.433 0.409 0.438 0.452 0.406]
mean Macro-F1:     0.428 +/- 0.018
held-out test Macro-F1 (from step 3): 0.417


## Step 5: One shared metrics function


In [ ]:
from common import load_split, scaled_pipeline as common_scaled_pipeline, full_report

X_train_c, X_test_c, y_train_c, y_test_c = load_split()
assert X_train.equals(X_train_c) and X_test.equals(X_test_c)
assert y_train.equals(y_train_c) and y_test.equals(y_test_c)
print("common.load_split() matches this notebook's split exactly.")

ref_model = common_scaled_pipeline(LogisticRegression(max_iter=500))
ref_model.fit(X_train, y_train)
y_pred = ref_model.predict(X_test)

summary, cm, pc = full_report("Logistic Regression (reference)", y_test, y_pred)
summary

In [8]:
cm

,pred_1,pred_2,pred_3,pred_4,pred_5
true_1,156,492,60,6,0
true_2,115,980,285,20,4
true_3,46,549,500,87,11
true_4,2,78,202,176,40
true_5,0,4,27,101,59


In [9]:
pc.round(3)

,precision,recall,f1-score,support
1,0.489,0.218,0.302,714.0
2,0.466,0.698,0.559,1404.0
3,0.466,0.419,0.441,1193.0
4,0.451,0.353,0.396,498.0
5,0.518,0.309,0.387,191.0
